# 03b — Verificación del dataset analítico v2 — OULAD

**Propósito.** Comprobar sobre `dataset_analitico_oulad_v2.parquet` las tres cosas que las
correcciones de la v2 debían resolver, y cerrar la única decisión de variables que quedó
abierta.

Este notebook **no reemplaza** a `03_analisis_estadistico.ipynb`. Aquel es el diagnóstico del
dataset v1 y es lo que justifica las correcciones; re-ejecutarlo sobre el v2 destruiría la
evidencia que sustenta el informe estadístico. Aquí solo se verifica que las correcciones
surtieron efecto.

**Contenido**

1. Carga del v2 y definición del conjunto de predictoras (separación de las variables de futuro).
2. Redundancias exactas: ¿queda alguna además de la identidad de quincenas?
3. Multicolinealidad: VIF y número de condición, con y sin `total_clics`.
4. Decisión sobre `coef_variacion_clics` reconstruida: Mann–Whitney y tamaño de efecto.
5. Conjunto multivariado sin la dependencia funcional (corrige el ACP y la Mahalanobis robusta).

In [1]:
import sys, pandas as pd, numpy as np
from pathlib import Path
from scipy import stats

pd.set_option('display.max_columns', None); pd.set_option('display.width', 160)
PROCESSED_DIR = Path('../data/processed')
FIGS_DIR = Path('../reportes/figs'); FIGS_DIR.mkdir(parents=True, exist_ok=True)

import scipy, sklearn, statsmodels
print('python', sys.version.split()[0], '| pandas', pd.__version__, '| numpy', np.__version__,
      '| scipy', scipy.__version__, '| sklearn', sklearn.__version__,
      '| statsmodels', statsmodels.__version__)

df = pd.read_parquet(PROCESSED_DIR / 'dataset_analitico_oulad_v2.parquet')
print(f'\nDataset v2: {df.shape[0]:,} inscripciones x {df.shape[1]} columnas')
assert df.shape == (32593, 34), f'Se esperaba (32593, 34), hay {df.shape}'

python 3.11.9 | pandas 2.2.2 | numpy 1.26.4 | scipy 1.17.1 | sklearn 1.5.0 | statsmodels 0.14.6

Dataset v2: 32,593 inscripciones x 34 columnas


## 1. Conjunto de predictoras

Seis columnas del dataset **no pueden entrar al modelo**: incorporan información posterior al
cierre de la ventana de observación o son identificadores. Dejarlas produciría fuga de
información y un desempeño optimista que no se sostiene en uso real.

| Columna | Por qué se excluye |
|---|---|
| `id_student` | identificador |
| `date_unregistration` | **es** la etiqueta |
| `momento_retiro` | derivada de la etiqueta |
| `final_result` | resultado al cierre del curso |
| `abandono_implicito_total` | se calcula sobre el curso completo |
| `sin_interaccion_vle` | se calcula sobre todo `studentVle`, no sobre la ventana |

`code_module` y `code_presentation` **sí** se conservan: son condiciones instruccionales
conocidas en el día 0 y el módulo es la categórica de mayor asociación con el abandono
(V de Cramér = 0,175), por lo que debe entrar como control o estrato.

In [2]:
EXCLUIR = ['id_student', 'date_unregistration', 'momento_retiro', 'final_result',
           'abandono_implicito_total', 'sin_interaccion_vle', 'abandono']

X = df.drop(columns=EXCLUIR)
y = df['abandono']

print(f'Predictoras: {X.shape[1]}')
assert X.shape[1] == 27, f'Se esperaban 27 predictoras, hay {X.shape[1]}'

NUM = X.select_dtypes(include=[np.number]).columns.tolist()
CAT = [c for c in X.columns if c not in NUM]
print(f'\nNumericas ({len(NUM)}):'); print(NUM)
print(f'\nCategoricas ({len(CAT)}):'); print(CAT)

Predictoras: 27

Numericas (19):
['date_registration', 'num_of_prev_attempts', 'studied_credits', 'total_clics', 'dias_activos', 'n_interacciones', 'promedio_clics_por_dia_activo', 'dias_desde_ultima_interaccion', 'coef_variacion_clics', 'semanas_activas', 'n_evaluaciones_entregadas', 'score_promedio_temprano', 'clics_quincena_1', 'clics_quincena_2', 'log_ratio_actividad', 'interaccion_pre_curso', 'n_interacciones_pre_curso', 'sin_actividad_ventana_temprana', 'sin_entregas_tempranas']

Categoricas (8):
['code_module', 'code_presentation', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']


## 2. Redundancias exactas

La v1 contenía tres relaciones deterministas entre predictoras. Dos se corrigieron
(`frecuencia_acceso` ≡ `dias_activos`/28 y `semanas_inactivas` ≡ 4 − `semanas_activas`); la
tercera, `total_clics` ≡ `clics_quincena_1` + `clics_quincena_2`, se conserva de forma
deliberada porque las tres variables tienen lectura sustantiva distinta, y se resuelve en el
modelado reteniendo un representante por bloque.

Se comprueba que no queda ninguna otra. El criterio es |ρ| > 0,999 sobre Spearman, más la
verificación algebraica directa de la identidad de quincenas.

In [3]:
corr_s = X[NUM].corr(method='spearman')
pares = [(corr_s.index[i], corr_s.columns[j], corr_s.iloc[i, j])
         for i in range(len(corr_s)) for j in range(i + 1, len(corr_s))
         if abs(corr_s.iloc[i, j]) > 0.999]

print('Pares con |rho| > 0,999:')
if pares:
    for a, b, r in pares: print(f'   {a} ~ {b}: rho = {r:.4f}')
else:
    print('   ninguno')

ident = np.allclose(df.total_clics, df.clics_quincena_1 + df.clics_quincena_2)
print(f'\ntotal_clics == q1 + q2 (identidad conocida y documentada): {ident}')

print('\nPares con 0,90 < |rho| <= 0,999 (colinealidad severa, no exacta):')
altos = [(corr_s.index[i], corr_s.columns[j], corr_s.iloc[i, j])
         for i in range(len(corr_s)) for j in range(i + 1, len(corr_s))
         if 0.90 < abs(corr_s.iloc[i, j]) <= 0.999]
for a, b, r in sorted(altos, key=lambda t: -abs(t[2])): print(f'   {a} ~ {b}: rho = {r:.3f}')
if not altos: print('   ninguno')

Pares con |rho| > 0,999:
   ninguno

total_clics == q1 + q2 (identidad conocida y documentada): True

Pares con 0,90 < |rho| <= 0,999 (colinealidad severa, no exacta):
   total_clics ~ n_interacciones: rho = 0.968
   dias_activos ~ n_interacciones: rho = 0.949
   total_clics ~ clics_quincena_2: rho = 0.926
   dias_activos ~ coef_variacion_clics: rho = -0.923
   total_clics ~ dias_activos: rho = 0.913
   total_clics ~ clics_quincena_1: rho = 0.911
   n_interacciones ~ clics_quincena_1: rho = 0.906


## 3. Multicolinealidad: VIF y número de condición

Dos precisiones metodológicas sobre el diagnóstico de la v1:

- **Con dependencia lineal exacta el VIF no está definido**, no es "infinito": la matriz de
  diseño es singular y $R^2_j = 1$ hace que $1/(1-R^2_j)$ no exista. Lo que devuelve la
  implementación es un valor numéricamente enorme, artefacto del redondeo.
- El VIF de la v1 se calculó sobre 20 011 **casos completos** (borrado por lista). Como los
  faltantes son MNAR informativos, esa subpoblación no es representativa. Aquí se calcula por
  bloques sobre la subpoblación donde cada variable está definida, y se reporta además el
  número de condición de la matriz estandarizada, que es el diagnóstico correcto ante
  singularidad.

Regla de lectura: VIF > 5 indica colinealidad apreciable; número de condición > 30, un
problema de acondicionamiento (Johnson & Wichern, *Applied Multivariate Statistical Analysis*,
6.ª ed., Pearson, 2007, §7.6).

In [4]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

def diagnostico(cols, etiqueta):
    sub = X[cols].dropna()
    Z = StandardScaler().fit_transform(sub)
    vifs = pd.Series([variance_inflation_factor(Z, i) for i in range(Z.shape[1])],
                     index=cols).sort_values(ascending=False)
    cond = np.linalg.cond(Z)
    rango = np.linalg.matrix_rank(Z)
    print(f'\n--- {etiqueta} | n = {len(sub):,} | p = {len(cols)} ---')
    print(vifs.round(2).to_string())
    print(f'numero de condicion = {cond:,.1f} | rango = {rango} de {len(cols)}'
          f'{"   <-- MATRIZ SINGULAR" if rango < len(cols) else ""}')
    return vifs, cond

INTERACCION = ['total_clics', 'dias_activos', 'n_interacciones', 'promedio_clics_por_dia_activo',
               'dias_desde_ultima_interaccion', 'semanas_activas', 'coef_variacion_clics',
               'clics_quincena_1', 'clics_quincena_2', 'log_ratio_actividad']

diagnostico(INTERACCION, 'Interaccion, CON total_clics (identidad q1+q2 presente)')
diagnostico([c for c in INTERACCION if c != 'total_clics'],
            'Interaccion, SIN total_clics (decision adoptada para modelos lineales)')

d:\proyecto-grado\prediccion-desercion-oulad\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



--- Interaccion, CON total_clics (identidad q1+q2 presente) | n = 27,886 | p = 10 ---
total_clics                        inf
clics_quincena_1                   inf
clics_quincena_2                   inf
n_interacciones                  10.32
dias_activos                      8.80
coef_variacion_clics              5.15
semanas_activas                   3.98
dias_desde_ultima_interaccion     3.70
promedio_clics_por_dia_activo     3.12
log_ratio_actividad               2.30
numero de condicion = 1,744,375,239,348,428.5 | rango = 9 de 10   <-- MATRIZ SINGULAR

--- Interaccion, SIN total_clics (decision adoptada para modelos lineales) | n = 27,886 | p = 9 ---
n_interacciones                  10.32
dias_activos                      8.80
coef_variacion_clics              5.15
clics_quincena_2                  5.01
clics_quincena_1                  4.60
semanas_activas                   3.98
dias_desde_ultima_interaccion     3.70
promedio_clics_por_dia_activo     3.12
log_ratio_actividad     

(n_interacciones                  10.318685
 dias_activos                      8.795608
 coef_variacion_clics              5.154892
 clics_quincena_2                  5.012577
 clics_quincena_1                  4.595891
 semanas_activas                   3.981197
 dias_desde_ultima_interaccion     3.698788
 promedio_clics_por_dia_activo     3.120751
 log_ratio_actividad               2.302111
 dtype: float64,
 8.803069877402969)

In [5]:
# Conjunto completo de numericas, con la decision adoptada aplicada
RETENIDAS = [c for c in NUM if c != 'total_clics']
vifs, cond = diagnostico(RETENIDAS, 'Todas las numericas retenidas')

print('\nVariables con VIF > 5:')
altas = vifs[vifs > 5]
print(altas.round(2).to_string() if len(altas) else '   ninguna')


--- Todas las numericas retenidas | n = 20,010 | p = 18 ---
n_interacciones                   9.84
dias_activos                      8.21
clics_quincena_2                  5.29
clics_quincena_1                  4.93
coef_variacion_clics              4.34
promedio_clics_por_dia_activo     4.10
semanas_activas                   2.86
dias_desde_ultima_interaccion     2.35
log_ratio_actividad               2.02
n_interacciones_pre_curso         1.68
interaccion_pre_curso             1.20
n_evaluaciones_entregadas         1.20
score_promedio_temprano           1.08
num_of_prev_attempts              1.04
studied_credits                   1.04
date_registration                 1.02
sin_actividad_ventana_temprana     NaN
sin_entregas_tempranas             NaN
numero de condicion = inf | rango = 16 de 18   <-- MATRIZ SINGULAR

Variables con VIF > 5:
n_interacciones     9.84
dias_activos        8.21
clics_quincena_2    5.29


d:\proyecto-grado\prediccion-desercion-oulad\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


## 4. Decisión sobre `coef_variacion_clics`

En la v1 la variable se calculaba solo sobre los días con actividad, de modo que —por
construcción— no podía capturar la irregularidad producida por la inactividad, que es
justamente la señal buscada. Su ausencia de poder discriminante (Mann–Whitney *p* = 0,058;
*r*<sub>rb</sub> = 0,016) era atribuible a esa construcción. La v2 la calcula sobre el vector
completo de 28 días, rellenando con ceros los días sin actividad.

**Contraste.** Con $X_{\text{ab}}$ y $X_{\text{no}}$ los valores en cada clase:

$$H_0:\; P(X_{\text{ab}} > X_{\text{no}}) + \tfrac12 P(X_{\text{ab}} = X_{\text{no}}) = \tfrac12
\qquad\text{vs.}\qquad H_1:\; \neq \tfrac12,$$

con $\alpha = 0,05$. El enunciado incluye el término de empates, indispensable con variables
discretas fuertemente empatadas como `dias_activos` o `n_evaluaciones_entregadas`; el informe
estadístico lo omitía.

**Tamaño de efecto.** $r_{\text{rb}} = 2U/(n_1 n_2) - 1$, que coincide con la $\delta$ de Cliff
y equivale a $P(X_{\text{ab}} > X_{\text{no}}) - P(X_{\text{ab}} < X_{\text{no}})$. Como `scipy`
asigna rangos medios a los empates, el estadístico $U$ ya los incorpora y el estimador queda
corregido sin ajuste adicional; la corrección por empates de la varianza también la aplica
`scipy` en la aproximación asintótica.

**Advertencia de interpretación.** Cada contraste se realiza sobre los casos con valor
observado. Como el faltante es MNAR informativo, ese condicionamiento excluye precisamente a
la subpoblación de mayor riesgo (79,12 % de abandono entre quienes no tienen actividad). Los
tamaños de efecto de esta tabla son, por tanto, **condicionales a la presencia del dato** y no
deben leerse como el poder discriminante total de la variable, que incluye la señal del propio
indicador de faltante.

In [6]:
def contraste(v):
    s = df[[v, 'abandono']].dropna()
    a = s.loc[s.abandono == 1, v].values
    b = s.loc[s.abandono == 0, v].values
    U, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    r_rb = 2 * U / (len(a) * len(b)) - 1
    empates = 1 - s[v].nunique() / len(s)
    return dict(variable=v, n=len(s), n_ab=len(a), n_no=len(b),
                mediana_ab=np.median(a), mediana_no=np.median(b),
                U=U, p=p, r_rb=r_rb, pct_empatados=empates)

tabla = pd.DataFrame([contraste(v) for v in NUM])
tabla['abs_r'] = tabla.r_rb.abs()
tabla = tabla.sort_values('abs_r', ascending=False).drop(columns='abs_r').reset_index(drop=True)

fmt = tabla.copy()
fmt['U'] = fmt.U.map(lambda x: f'{x:.3e}')
fmt['p'] = fmt.p.map(lambda x: '< 0,001' if x < 1e-3 else f'{x:.3f}')
fmt['r_rb'] = fmt.r_rb.round(3)
fmt['pct_empatados'] = (fmt.pct_empatados * 100).round(1)
print(fmt.to_string(index=False))

                      variable     n  n_ab  n_no  mediana_ab  mediana_no         U       p   r_rb  pct_empatados
                  dias_activos 32593 10072 22521    3.000000   10.000000 6.069e+07 < 0,001 -0.465           99.9
               n_interacciones 32593 10072 22521   15.000000   60.000000 6.203e+07 < 0,001 -0.453           98.7
                   total_clics 32593 10072 22521   36.000000  184.000000 6.259e+07 < 0,001 -0.448           95.0
              clics_quincena_2 32593 10072 22521    4.000000   87.000000 6.379e+07 < 0,001 -0.438           96.9
              clics_quincena_1 32593 10072 22521   11.000000   80.000000 6.598e+07 < 0,001 -0.418           96.9
        sin_entregas_tempranas 32593 10072 22521    1.000000    0.000000 1.523e+08 < 0,001  0.343          100.0
sin_actividad_ventana_temprana 32593 10072 22521    0.000000    0.000000 1.504e+08 < 0,001  0.326          100.0
     n_interacciones_pre_curso 32593 10072 22521    2.000000   12.000000 7.904e+07 < 0,001 -0.30

In [7]:
# Comparacion directa v1 vs v2 para la variable en decision
fila = tabla[tabla.variable == 'coef_variacion_clics'].iloc[0]

print('coef_variacion_clics — v1 (CV entre dias activos) vs v2 (CV sobre los 28 dias)\n')
print(f'{"":26s}{"v1":>14s}{"v2":>14s}')
print(f'{"n con valor observado":26s}{26443:>14,}{int(fila.n):>14,}')
print(f'{"mediana (abandono)":26s}{0.94:>14.3f}{fila.mediana_ab:>14.3f}')
print(f'{"mediana (no abandono)":26s}{0.92:>14.3f}{fila.mediana_no:>14.3f}')
print(f'{"p (Mann-Whitney)":26s}{0.058:>14.3f}{fila.p:>14.3g}')
print(f'{"r_rb (delta de Cliff)":26s}{0.016:>14.3f}{fila.r_rb:>14.3f}')

print('\nCriterio de decision: se conserva si |r_rb| supera de forma clara el umbral de efecto')
print('despreciable y no es redundante con las variables ya retenidas.')
print(f'\n|r_rb| = {abs(fila.r_rb):.3f}')
print('Correlacion de Spearman con las demas variables de interaccion:')
print(corr_s.loc['coef_variacion_clics',
                 [c for c in INTERACCION if c != 'coef_variacion_clics']].round(3).to_string())

coef_variacion_clics — v1 (CV entre dias activos) vs v2 (CV sobre los 28 dias)

                                      v1            v2
n con valor observado             26,443        27,886
mediana (abandono)                 0.940         2.555
mediana (no abandono)              0.920         2.101
p (Mann-Whitney)                   0.058     9.91e-169
r_rb (delta de Cliff)              0.016         0.228

Criterio de decision: se conserva si |r_rb| supera de forma clara el umbral de efecto
despreciable y no es redundante con las variables ya retenidas.

|r_rb| = 0.228
Correlacion de Spearman con las demas variables de interaccion:
total_clics                     -0.771
dias_activos                    -0.923
n_interacciones                 -0.865
promedio_clics_por_dia_activo   -0.327
dias_desde_ultima_interaccion    0.630
semanas_activas                 -0.727
clics_quincena_1                -0.749
clics_quincena_2                -0.690
log_ratio_actividad             -0.008


### Registro de la decisión

Complete esta celda con la lectura de los resultados anteriores. La decisión debe quedar
escrita aquí y trasladarse al capítulo 4 del documento, no inferirse de la salida.

- **Si |r_rb| sigue siendo despreciable** (orden de 0,02 o menor): la variable se excluye, y
  ahora con fundamento —la reconstrucción descarta que la ausencia de señal fuera un artefacto
  de construcción—.
- **Si |r_rb| aumenta de forma apreciable**: se conserva, y debe corregirse el Cuadro 8 del
  informe estadístico, que la registra como "candidata a exclusión".

> **Decisión adoptada:** _(escribir aquí)_

## 5. Conjunto multivariado sin la dependencia funcional

El análisis multivariado de la v1 (Mahalanobis robusta con MCD, contraste de Hotelling y ACP)
se calculó sobre seis variables entre las que existe una dependencia funcional:

$$\texttt{promedio\_clics\_por\_dia\_activo} = \frac{\texttt{total\_clics}}{\texttt{dias\_activos}}
\;\Longrightarrow\;
\log(\texttt{total}) = \log(\texttt{promedio}) + \log(\texttt{dias\_activos}).$$

La transformación `log1p` y el hecho de que `dias_activos` entrara sin transformar rompen la
identidad exacta, pero dejan una casi-singularidad cuya firma es el 0,3 % de varianza del sexto
componente principal. Eso compromete la inversión de la matriz de covarianzas que exigen la
distancia de Mahalanobis y el estadístico de Hotelling.

Se rehace el conjunto suprimiendo `total_clics`, que es el término derivable de los otros dos.

In [8]:
from sklearn.decomposition import PCA

sub = df[df.dias_activos > 0].copy()
sub['log1p_total_clics'] = np.log1p(sub.total_clics)
sub['log1p_prom_clics']  = np.log1p(sub.promedio_clics_por_dia_activo)

MV_v1 = ['log1p_total_clics', 'dias_activos', 'log1p_prom_clics',
         'dias_desde_ultima_interaccion', 'semanas_activas', 'log_ratio_actividad']
MV_v2 = [c for c in MV_v1 if c != 'log1p_total_clics']

for nombre, cols in [('v1 (con la dependencia funcional)', MV_v1),
                     ('v2 (sin total_clics)', MV_v2)]:
    Z = StandardScaler().fit_transform(sub[cols].dropna())
    pca = PCA().fit(Z)
    ev = pca.explained_variance_ratio_ * 100
    print(f'\n--- Conjunto multivariado {nombre} | n = {Z.shape[0]:,} | p = {len(cols)} ---')
    print('varianza explicada (%):', np.round(ev, 2))
    print(f'acumulada 2 componentes: {ev[:2].sum():.1f} %')
    print(f'numero de condicion: {np.linalg.cond(Z):,.1f}'
          f' | ultimo componente: {ev[-1]:.2f} %')


--- Conjunto multivariado v1 (con la dependencia funcional) | n = 27,886 | p = 6 ---
varianza explicada (%): [58.23 19.04 14.01  5.66  2.78  0.27]
acumulada 2 componentes: 77.3 %
numero de condicion: 14.6 | ultimo componente: 0.27 %

--- Conjunto multivariado v2 (sin total_clics) | n = 27,886 | p = 5 ---
varianza explicada (%): [52.82 21.78 15.32  6.76  3.32]
acumulada 2 componentes: 74.6 %
numero de condicion: 4.0 | ultimo componente: 3.32 %


In [9]:
# 3b. Diagnostico de la singularidad del conjunto retenido (n = 20,010)
sub = X[RETENIDAS].dropna()
print(f'casos completos: {len(sub):,} de {len(X):,}\n')

d = pd.DataFrame({'nunique': sub.nunique(), 'std': sub.std(),
                  'min': sub.min(), 'max': sub.max(),
                  'nunique_completo': X[RETENIDAS].nunique()})
print(d.sort_values('nunique').to_string())

const = d.index[d['nunique'] == 1].tolist()
print('\nConstantes SOLO en los casos completos:', const or 'ninguna')

# Espectro: distingue varianza nula de colinealidad
Z = StandardScaler().fit_transform(sub)
s = np.linalg.svd(Z, compute_uv=False)
print('\nvalores singulares:', np.round(s, 6))
print(f'nulos (< 1e-8): {(s < 1e-8).sum()} | rango = {np.linalg.matrix_rank(Z)} de {len(RETENIDAS)}')

# Recalculo sobre las no degeneradas
NO_DEG = [c for c in RETENIDAS if c not in const]
if const:
    diagnostico(NO_DEG, 'Numericas retenidas, sin las degeneradas en casos completos')

casos completos: 20,010 de 32,593

                                nunique         std         min          max  nunique_completo
sin_entregas_tempranas                1    0.000000    0.000000     0.000000                 2
sin_actividad_ventana_temprana        1    0.000000    0.000000     0.000000                 2
interaccion_pre_curso                 2    0.341607    0.000000     1.000000                 2
n_evaluaciones_entregadas             4    0.425027    1.000000     4.000000                 4
semanas_activas                       4    0.750886    1.000000     4.000000                 4
num_of_prev_attempts                  6    0.449879    0.000000     5.000000                 7
dias_desde_ultima_interaccion        28    3.485910    0.000000    27.000000                28
dias_activos                         28    6.820889    1.000000    28.000000                29
studied_credits                      45   37.190525   30.000000   630.000000                61
score_promedio_

In [10]:
# 3c. Multicolinealidad sobre la poblacion completa (n = 32,593)
# Unicas numericas sin faltantes; se excluye total_clics por la identidad q1 + q2.
COMPLETAS = ['dias_activos', 'n_interacciones', 'clics_quincena_1', 'clics_quincena_2',
             'n_interacciones_pre_curso', 'interaccion_pre_curso',
             'sin_actividad_ventana_temprana', 'sin_entregas_tempranas',
             'studied_credits', 'num_of_prev_attempts']

falt = X[COMPLETAS].isna().sum()
assert falt.sum() == 0, f'Hay faltantes:\n{falt[falt > 0]}'
assert len(COMPLETAS) == 10

# (a) Las indicadoras, ¿son funcion determinista de otra predictora?
print('sin_actividad_ventana_temprana == (dias_activos == 0):',
      X.sin_actividad_ventana_temprana.astype(bool).equals(X.dias_activos.eq(0)))
print('sin_entregas_tempranas == (n_evaluaciones_entregadas nula o 0):',
      X.sin_entregas_tempranas.astype(bool).equals(
          X.n_evaluaciones_entregadas.isna() | X.n_evaluaciones_entregadas.eq(0)))
print(X[COMPLETAS].nunique().to_string())

# (b) VIF y numero de condicion
vifs_c, cond_c = diagnostico(COMPLETAS, 'Numericas definidas en las 32,593 inscripciones')

# (c) Sin la indicadora de inactividad, si resulta redundante con dias_activos
diagnostico([c for c in COMPLETAS if c != 'sin_actividad_ventana_temprana'],
            'Idem, sin sin_actividad_ventana_temprana')

# (d) Spearman del bloque, para leer los VIF altos
print('\nSpearman (|rho| > 0,60) dentro del bloque:')
cs = X[COMPLETAS].corr(method='spearman')
for i in range(len(COMPLETAS)):
    for j in range(i + 1, len(COMPLETAS)):
        if abs(cs.iloc[i, j]) > 0.60:
            print(f'   {COMPLETAS[i]} ~ {COMPLETAS[j]}: {cs.iloc[i, j]:.3f}')

sin_actividad_ventana_temprana == (dias_activos == 0): True
sin_entregas_tempranas == (n_evaluaciones_entregadas nula o 0): True
dias_activos                        29
n_interacciones                    412
clics_quincena_1                  1026
clics_quincena_2                  1015
n_interacciones_pre_curso          259
interaccion_pre_curso                2
sin_actividad_ventana_temprana       2
sin_entregas_tempranas               2
studied_credits                     61
num_of_prev_attempts                 7

--- Numericas definidas en las 32,593 inscripciones | n = 32,593 | p = 10 ---
n_interacciones                   12.54
dias_activos                       6.29
clics_quincena_2                   3.41
clics_quincena_1                   3.25
n_interacciones_pre_curso          1.77
sin_actividad_ventana_temprana     1.76
sin_entregas_tempranas             1.62
interaccion_pre_curso              1.57
studied_credits                    1.07
num_of_prev_attempts               1.04
nu

## Síntesis

Complete tras la ejecución; es el texto que se traslada al capítulo 4 del documento.

| Verificación | Resultado |
|---|---|
| 34 columnas, 32 593 inscripciones | |
| 27 predictoras tras excluir las seis variables de futuro | |
| Redundancias exactas fuera de `total_clics ≡ q₁ + q₂` | |
| VIF sin `total_clics`: máximo y variables > 5 | |
| Número de condición del conjunto retenido | |
| `coef_variacion_clics`: *p*, *r*<sub>rb</sub> y decisión | |
| Conjunto multivariado: varianza del último componente | |

**Pendiente que este notebook no resuelve:** la definición del evento a predecir. Las tres rutas
y sus prevalencias (18,26 % / 30,90 % / 24,72 %) están calculadas en
`02_feature_engineering.ipynb`; la decisión corresponde a la dirección del proyecto y es
condición previa al Objetivo específico 3.